<a href="https://colab.research.google.com/github/JP-DSmith/woodb-agent-project/blob/feature%2Fsynthetic-datasets-fix/synthetic_data_pipeline_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Synthetic Data Pipeline: End-to-End Execution
This consolidated notebook contains only the successful, error-free cells. It authenticates with GitHub, clones the feature branch, provisions the database, generates the datasets, commits the results, and opens the pull request.

## Secure Authorization & Clean Branch Checkout

In [8]:
from google.colab import userdata
import subprocess

# 1. Authenticate GitHub CLI
token = userdata.get("GITHUB_PAT")
if not token:
    raise RuntimeError("GITHUB_PAT was not found in Colab Secrets.")

subprocess.run(["gh", "auth", "login", "--hostname", "github.com", "--git-protocol", "https", "--with-token"], input=token, text=True, check=True)
subprocess.run(["gh", "auth", "setup-git"], check=True)

# 2. Wipe previous instances and pull directly from dev
!rm -rf /content/woodb-agent-project
!git clone -b dev https://github.com/JP-DSmith/woodb-agent-project.git /content/woodb-agent-project
!git -C /content/woodb-agent-project remote set-url origin https://github.com/JP-DSmith/woodb-agent-project.git

print("✅ Authenticated and cloned fresh from dev branch.")

Cloning into '/content/woodb-agent-project'...
remote: Enumerating objects: 320, done.
remote: Counting objects: 100% (120/120), done.
remote: Compressing objects: 100% (113/113), done.
remote: Total 320 (delta 49), reused 20 (delta 7), pack-reused 200 (from 1)
Receiving objects: 100% (320/320), 308.99 KiB | 3.55 MiB/s, done.
Resolving deltas: 100% (104/104), done.
✅ Authenticated and cloned fresh from dev branch.


## World File Sanity Check

In [9]:
import pandas as pd
from pathlib import Path

world_dir = Path('/content/woodb-agent-project/prototypes/cusma_qualification/World')

# Mapping each file to the primary column we want to sample
files_to_check = {
    'skus.csv': 'sku',
    'bom_lines.csv': 'sku',
    'suppliers.csv': 'supplier_id',
    'orders.csv': 'po_number',
    'customers.csv': 'company',
    'ground_truth.csv': 'sku',
    'order_truth.csv': 'po_number'
}

print("--- WORLD FILES SANITY CHECK ---")
for filename, id_col in files_to_check.items():
    try:
        file_path = world_dir / filename
        df = pd.read_csv(file_path)

        # Fallback to the first column if our expected ID column is missing
        sample_col = id_col if id_col in df.columns else df.columns[0]
        samples = df[sample_col].dropna().head(3).tolist()

        print(f"✅ {filename:<18} | {len(df):>4} records | Samples ({sample_col}): {samples}")
    except FileNotFoundError:
        print(f"❌ {filename:<18} | ERROR: File not found in {world_dir}")
    except Exception as e:
        print(f"❌ {filename:<18} | ERROR: {e}")

--- WORLD FILES SANITY CHECK ---
✅ skus.csv           |   44 records | Samples (sku): ['NB-0101', 'NB-0101-V1', 'NB-0101-V2']
✅ bom_lines.csv      |  376 records | Samples (sku): ['NB-0101', 'NB-0101', 'NB-0101']
✅ suppliers.csv      |   46 records | Samples (supplier_id): ['SUP-010', 'SUP-011', 'SUP-012']
✅ orders.csv         |   44 records | Samples (po_number): ['PO-8001', 'PO-8002', 'PO-8003']
✅ customers.csv      |    6 records | Samples (company): ['Lakehead Workwear Supply', 'Green Mountain Safety Outfitters', 'Cascade Marine & Outdoor']
✅ ground_truth.csv   |   44 records | Samples (sku): ['NB-0101', 'NB-0101-V1', 'NB-0101-V2']
✅ order_truth.csv    |   44 records | Samples (po_number): ['PO-8001', 'PO-8002', 'PO-8003']


## The Hardened Combinatorial Pipeline

In [10]:
import random
import json
from pathlib import Path
import pandas as pd

REPO_ROOT = Path("/content/woodb-agent-project")
WORLD_FOLDER = REPO_ROOT / "prototypes" / "cusma_qualification" / "World"
DATA_DIR = REPO_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

class UltimateQCGenerator:
    def __init__(self):
        self.skus = pd.read_csv(WORLD_FOLDER / "skus.csv").to_dict('records')
        self.bom_lines = pd.read_csv(WORLD_FOLDER / "bom_lines.csv").to_dict('records')
        self.suppliers = pd.read_csv(WORLD_FOLDER / "suppliers.csv").to_dict('records')
        self.orders = pd.read_csv(WORLD_FOLDER / "orders.csv").to_dict('records')
        self.customers = pd.read_csv(WORLD_FOLDER / "customers.csv").to_dict('records')
        self.ground_truth = pd.read_csv(WORLD_FOLDER / "ground_truth.csv")
        self.order_truth = pd.read_csv(WORLD_FOLDER / "order_truth.csv")

        self.hs_codes = self.ground_truth['hs_code'].dropna().unique().tolist() if 'hs_code' in self.ground_truth.columns else ['6403.99']

        self.class_mix = {
            "qualification_check": 0.20, "sourcing_whatif": 0.15, "duty_cost": 0.15,
            "certification_draft": 0.12, "document_intake": 0.12, "rules_info": 0.10,
            "escalate_human": 0.08, "out_of_scope": 0.08
        }

        self.personas = ["Customs Broker", "Supply Chain Manager", "Customer Service", "Retail Buyer", "Compliance Officer", "Logistics Analyst", "Factory Manager", "Auditor"]
        self.prefixes = ["", "Hi team, ", "Quick question: ", "Urgent: ", "Hello, ", "Please advise: "]
        self.random_items = ['PVC boat fender', 'rubber boat bumpers', 'my personal tax return', 'my HR benefits']
        self.cities = ['Montreal', 'Toronto', 'Vancouver', 'New York']

        # Flattened English-only builders dictionary
        self.builders = {
            "qualification_check": lambda: f"{random.choice(self.prefixes)}Does {random.choice(['the {sku}', 'our {boot_name}', 'sku {sku}'])} qualify under CUSMA?",
            "sourcing_whatif": lambda: f"{random.choice(self.prefixes)}If we switch the {random.choice(['{component}', 'our component'])} from {random.choice(['{new_country}', '{new_supplier}'])} instead, does {random.choice(['{sku}', '{boot_name}'])} still qualify?",
            "duty_cost": lambda: f"{random.choice(self.prefixes)}How much duty for {random.choice(['{boot_name}', 'order {po_number}', '{sku}'])}?",
            "certification_draft": lambda: f"{random.choice(self.prefixes)}Please draft a CUSMA certificate for {random.choice(['{customer_company}', '{boot_name}'])}.",
            "document_intake": lambda: f"{random.choice(self.prefixes)}Here is the new origin declaration for {random.choice(['{new_supplier}', '{sku}'])}.",
            "rules_info": lambda: f"{random.choice(self.prefixes)}What is the tariff shift rule for {random.choice(['{boot_name}', '{sku}'])}?",
            "escalate_human": lambda: f"{random.choice(self.prefixes)}Customs is auditing {random.choice(['{sku}', '{boot_name}', '{po_number}'])} due to compliance review.",
            "out_of_scope": lambda: f"{random.choice(self.prefixes)}What is the duty on {random.choice(self.random_items)}?"
        }

    def _get_boot_family(self, sku):
        parts = str(sku).split('-')
        return f"{parts[0]}-{parts[1]}" if len(parts) >= 2 else str(sku)

    def generate_split_dataset(self, train_size=2000, val_size=500, test_size=500):
        all_skus = list(set([s['sku'] for s in self.skus]))
        families = list(set([self._get_boot_family(sku) for sku in all_skus]))
        random.shuffle(families)

        split_idx = int(len(families) * 0.8)
        train_val_families = set(families[:split_idx])
        test_families = set(families[split_idx:])

        train_val_skus = [s for s in self.skus if self._get_boot_family(s['sku']) in train_val_families]
        test_skus = [s for s in self.skus if self._get_boot_family(s['sku']) in test_families]

        global_seen_queries = set()
        reject_log = []

        def fill_split(target_size, allowed_skus, is_test=False):
            split_dataset = []
            target_counts = {intent: int(target_size * weight) for intent, weight in self.class_mix.items()}
            current_counts = {intent: 0 for intent in self.class_mix.keys()}
            attempts = 0

            while len(split_dataset) < target_size:
                available_intents = [k for k, v in current_counts.items() if v < target_counts[k]]
                if not available_intents:
                    available_intents = list(self.class_mix.keys())
                    if len(split_dataset) >= target_size: break

                intent = random.choice(available_intents)
                sample_sku = random.choice(allowed_skus)

                sku_bom_options = [bom for bom in self.bom_lines if bom['sku'] == sample_sku['sku']]
                sample_bom = random.choice(sku_bom_options) if sku_bom_options else random.choice(self.bom_lines)

                alts = [s for s in self.suppliers if s['component'] == sample_bom['component'] and s['supplier_id'] != sample_bom['supplier_id']]
                if not alts:
                    continue
                sample_supplier = random.choice(alts)
                new_country = sample_supplier['country']

                sample_order = random.choice(self.orders)
                po_number = sample_order.get('po_number', 'PO-UNKNOWN')
                sample_customer = random.choice(self.customers)
                customer_company = sample_customer.get('company', 'UNKNOWN CO')

                raw_template = self.builders[intent]()
                query = raw_template.format(
                    sku=sample_sku['sku'],
                    boot_name=sample_sku.get('style_name', ''),
                    component=str(sample_bom['component']).lower(),
                    new_country=new_country,
                    new_supplier=sample_supplier.get('name', ''),
                    po_number=po_number,
                    customer_company=customer_company
                ).replace("  ", " ").strip()

                # Strictly English-only enforcement
                lang = "en"

                if query in global_seen_queries:
                    reject_log.append({"query": query, "intent": intent, "reject_reason": "Duplicate query"})
                    attempts += 1
                    if attempts > 8000: break
                else:
                    global_seen_queries.add(query)

                    answer = "N/A"
                    if intent == "duty_cost" or intent == "escalate_human":
                        match = self.order_truth[self.order_truth['po_number'] == po_number]
                        if not match.empty:
                            answer = match.iloc[0].get('verdict_on_ship_date', 'Unknown Order Verdict')
                    elif intent != "out_of_scope":
                        match = self.ground_truth[self.ground_truth['sku'] == sample_sku['sku']]
                        if not match.empty:
                            answer = match.iloc[0].get('verdict', 'Unknown CUSMA Verdict')

                    record = {
                        "query": query,
                        "intent": intent,
                        "language": lang,
                        "sku": sample_sku['sku'],
                        "po_number": po_number,
                        "persona": random.choice(self.personas),
                        "tier": random.choice(["T1", "T2", "T3"]),
                        "answer": answer
                    }

                    split_dataset.append(record)
                    current_counts[intent] += 1
                    attempts = 0

            return split_dataset

        train_data = fill_split(train_size, train_val_skus)
        val_data = fill_split(val_size, train_val_skus)
        test_data = fill_split(test_size, test_skus, is_test=True)

        return train_data, val_data, test_data, reject_log

if __name__ == "__main__":
    random.seed(42)
    generator = UltimateQCGenerator()
    train_dataset, val_dataset, test_dataset, reject_log = generator.generate_split_dataset(train_size=2000, val_size=500, test_size=500)

    with open(DATA_DIR / "train_dataset.jsonl", "w") as f:
        for item in train_dataset: f.write(json.dumps(item) + "\n")
    with open(DATA_DIR / "val_dataset.jsonl", "w") as f:
        for item in val_dataset: f.write(json.dumps(item) + "\n")
    with open(DATA_DIR / "test_holdout_human_reviewed.jsonl", "w") as f:
        for item in test_dataset: f.write(json.dumps(item) + "\n")
    with open(DATA_DIR / "reject_log.jsonl", "w") as f:
        for item in reject_log: f.write(json.dumps(item) + "\n")

    print(f"✅ Generated {len(train_dataset)} train, {len(val_dataset)} val, and {len(test_dataset)} test 100% English records with exact verdicts.")

✅ Generated 1852 train, 400 val, and 460 test 100% English records with exact verdicts.


## Data Integrity Test

In [11]:
import pandas as pd
import json
from pathlib import Path

DATA_DIR = Path("/content/woodb-agent-project/data")

def load_jsonl(file_path):
    return pd.DataFrame([json.loads(line) for line in open(file_path, 'r')])

print("--- PIPELINE DATA INTEGRITY & LEAKAGE TEST ---")

try:
    train_df = load_jsonl(DATA_DIR / "train_dataset.jsonl")
    val_df = load_jsonl(DATA_DIR / "val_dataset.jsonl")
    test_df = load_jsonl(DATA_DIR / "test_holdout_human_reviewed.jsonl")
    reject_df = load_jsonl(DATA_DIR / "reject_log.jsonl")
    print("✅ Files successfully loaded.\n")
except Exception as e:
    print(f"❌ Error loading files: {e}")
    exit()

# 1. Volume Check
print("1. Volume Check:")
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# 2. Boot Family Isolation (The most critical fix)
def get_family(sku):
    parts = str(sku).split('-')
    return f"{parts[0]}-{parts[1]}" if len(parts) >= 2 else str(sku)

train_families = set(train_df['sku'].apply(get_family))
val_families = set(val_df['sku'].apply(get_family))
test_families = set(test_df['sku'].apply(get_family))

train_test_overlap = train_families.intersection(test_families)
val_test_overlap = val_families.intersection(test_families)

print("\n2. Boot Family Leakage Check (Zero overlap required):")
print(f"Test families in Train: {len(train_test_overlap)} -> {'✅' if len(train_test_overlap) == 0 else '❌ (LEAKAGE)'}")
print(f"Test families in Val: {len(val_test_overlap)} -> {'✅' if len(val_test_overlap) == 0 else '❌ (LEAKAGE)'}")

# 3. Fake Data / Bracket Check
print("\n3. Fake Data / Formatting Check:")
fake_skus = pd.concat([train_df, val_df, test_df])['sku'].astype(str).str.contains('SKU-001|CK-9902').sum()
fake_ords = pd.concat([train_df, val_df, test_df])['po_number'].astype(str).str.contains('ORD-').sum()
bad_brackets = pd.concat([train_df, val_df, test_df])['query'].str.contains(r'\{|\}').sum()

print(f"Fake 'SKU-001' style SKUs found: {fake_skus} -> {'✅' if fake_skus == 0 else '❌'}")
print(f"Fake 'ORD-1000' style POs found: {fake_ords} -> {'✅' if fake_ords == 0 else '❌'}")
print(f"Unresolved {{brackets}} found: {bad_brackets} -> {'✅' if bad_brackets == 0 else '❌'}")

# 4. Language Check (T15 Prep)
print("\n4. Language Distribution (Test Set):")
print(test_df['language'].value_counts(normalize=True).round(3).to_string())

# 5. Output Samples
print("\n5. Random Query Sample (Checking new metadata):")
print(train_df.sample(2)[['intent', 'po_number', 'sku', 'answer', 'query']].to_string(index=False))

--- PIPELINE DATA INTEGRITY & LEAKAGE TEST ---
✅ Files successfully loaded.

1. Volume Check:
Train: 1852 | Val: 400 | Test: 460

2. Boot Family Leakage Check (Zero overlap required):
Test families in Train: 0 -> ✅
Test families in Val: 0 -> ✅

3. Fake Data / Formatting Check:
Fake 'SKU-001' style SKUs found: 0 -> ✅
Fake 'ORD-1000' style POs found: 0 -> ✅
Unresolved {brackets} found: 0 -> ✅

4. Language Distribution (Test Set):
language
en    1.0

5. Random Query Sample (Checking new metadata):
         intent po_number        sku           answer                                                        query
document_intake   PO-8028 NB-0413-V3     needs_review           Here is the new origin declaration for NB-0413-V3.
     rules_info   PO-8008 NB-0414-V3 does_not_qualify Please advise: What is the tariff shift rule for NB-0414-V3?


In [12]:
import pandas as pd
import json
from pathlib import Path

DATA_DIR = Path("/content/woodb-agent-project/data")

def load_jsonl(filename):
    path = DATA_DIR / filename
    if not path.exists():
        print(f"⚠️ {filename} not found.")
        return pd.DataFrame()
    return pd.DataFrame([json.loads(line) for line in open(path, 'r')])

print("==========================================")
print("       DATASET TEST & VERIFICATION        ")
print("==========================================")

train_df = load_jsonl("train_dataset.jsonl")
val_df = load_jsonl("val_dataset.jsonl")
test_df = load_jsonl("test_holdout_human_reviewed.jsonl")
reject_df = load_jsonl("reject_log.jsonl")

print(f"📁 train_dataset.jsonl : {len(train_df)} rows")
print(f"📁 val_dataset.jsonl   : {len(val_df)} rows")
print(f"📁 test_holdout.jsonl  : {len(test_df)} rows")
print(f"📁 reject_log.jsonl    : {len(reject_df)} rows")
print("------------------------------------------")

# 1. Intent Distribution Check (Train)
if not train_df.empty:
    print("\n📊 Training Set Intent Mix:")
    print(train_df['intent'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

# 2. Zero Leakage & Fake Data Assertion
if not train_df.empty and not test_df.empty:
    def get_family(sku):
        parts = str(sku).split('-')
        return f"{parts[0]}-{parts[1]}" if len(parts) >= 2 else str(sku)

    train_fam = set(train_df['sku'].apply(get_family))
    test_fam = set(test_df['sku'].apply(get_family))
    overlap = train_fam.intersection(test_fam)

    print(f"\n🔒 Boot Family Leakage Check: {len(overlap)} overlapping families -> {'✅ PASSED' if len(overlap) == 0 else '❌ FAILED'}")

    all_df = pd.concat([train_df, val_df, test_df])
    fake_skus = all_df['sku'].astype(str).str.contains('SKU-001|CK-9902').sum()
    fake_ords = all_df['po_number'].astype(str).str.contains('ORD-').sum()

    print(f"🛡️ Fake Data Check (SKU-001 / ORD-): {fake_skus + fake_ords} found -> {'✅ PASSED' if (fake_skus + fake_ords) == 0 else '❌ FAILED'}")

# 3. Sample Inspect
if not test_df.empty:
    print("\n🔍 Sample Test Record (Human-Reviewed Holdout):")
    sample = test_df.sample(1).iloc[0]
    print(json.dumps({
        "query": sample['query'],
        "intent": sample['intent'],
        "sku": sample['sku'],
        "po_number": sample['po_number'],
        "answer": sample['answer']
    }, indent=2))

print("==========================================")
print("✅ All test assertions completed successfully.")

       DATASET TEST & VERIFICATION        
📁 train_dataset.jsonl : 1852 rows
📁 val_dataset.jsonl   : 400 rows
📁 test_holdout.jsonl  : 460 rows
📁 reject_log.jsonl    : 31637 rows
------------------------------------------

📊 Training Set Intent Mix:
intent
qualification_check    21.6%
sourcing_whatif        16.2%
duty_cost              16.2%
document_intake        13.0%
certification_draft    12.3%
rules_info             10.8%
escalate_human          8.6%
out_of_scope            1.3%
Name: proportion, dtype: object

🔒 Boot Family Leakage Check: 0 overlapping families -> ✅ PASSED
🛡️ Fake Data Check (SKU-001 / ORD-): 0 found -> ✅ PASSED

🔍 Sample Test Record (Human-Reviewed Holdout):
{
  "query": "Hi team, Does our Value Knee (variant 3) qualify under CUSMA?",
  "intent": "qualification_check",
  "sku": "NB-0108-V3",
  "po_number": "PO-8037",
  "answer": "does_not_qualify"
}
✅ All test assertions completed successfully.


## Commit & Push

In [13]:
!git -C /content/woodb-agent-project checkout -b feature/synthetic-datasets-fix

Switched to a new branch 'feature/synthetic-datasets-fix'


In [15]:
# 1. Set your Git author identity locally
!git -C /content/woodb-agent-project config user.name "MGONDA7"
!git -C /content/woodb-agent-project config user.email "mukobigonda@gmail.com"

# 2. Stage the dataset files
!git -C /content/woodb-agent-project add data/train_dataset.jsonl data/val_dataset.jsonl data/test_holdout_human_reviewed.jsonl data/reject_log.jsonl

# 3. Commit the changes
!git -C /content/woodb-agent-project commit -m "fix: finalize English-only router datasets with exact World file verdicts"

# 4. Push to your feature branch
!git -C /content/woodb-agent-project push origin feature/synthetic-datasets-fix

[feature/synthetic-datasets-fix 75c7e3f] fix: finalize English-only router datasets with exact World file verdicts
 4 files changed, 34349 insertions(+), 4354 deletions(-)
remote: Permission to JP-DSmith/woodb-agent-project.git denied to MGONDA7.
fatal: unable to access 'https://github.com/JP-DSmith/woodb-agent-project.git/': The requested URL returned error: 403
